[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://github.com/codespaces)


<!-- gen:values.yml#weeks.04.notebook-title|Quest 04: The Messy Merger -->
# Week 4 Quest: The Messy Merger
<!-- /gen -->

*Two recently merged retailers keep sales ledgers that agree on nothing, and the board needs one combined revenue figure by Friday. You've been asked to help them!*

This [quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest) takes about 90 minutes. Everything you need is in this week's [Demonstration](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/04/Demonstration.ipynb). Work with your [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate) throughout.

**📼 This quest is recorded.** [SpecStory](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-specstory) writes a transcript of this session into `.specstory/history/` while you work. Your [AI Use Record](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-use-record) draws on that transcript. Before you start, check that SpecStory is recording. The [workstation guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/codespaces-start.md), section "Step 5: Check your record (SpecStory)", shows how.

## Stage 1: Briefing

*(about 10 minutes)*

> **From:** A. Partner, [Runnymede Analytics](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-runnymede-analytics)
> **To:** You ([Junior Analyst](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-analyst))
> **Re:** Hartley & Voss: One company, two ledgers, no numbers
>
> Hartley & Co (homeware, three shops) and Voss Retail (kitchenware, three shops) merged in December. It is July. According to the press <!-- plain-language: everyday release -->release, the integration is "going brilliantly". Nobody can state the combined company's monthly revenue. The two sales ledgers disagree about column names, date formats, and what counts as a row. Finance exported both and gave up.
>
> The board meets **Friday**. I need one table: **Combined monthly revenue, January to June 2026, in pounds**. I also need to know what you cleaned to get it, and what it cost. If we dropped or repaired anything, the board will ask *how much money* that <!-- plain-language: everyday decision -->decision moved.
>
> Your Associate will happily put these files together in one prompt. Combine them as someone who will be asked questions afterwards. On Friday, you will be.
>
> A.P.

**The business question:** What was Hartley & Voss's combined monthly revenue, January to June 2026?

**The deliverable:** One tidy table (month by month, in pounds). It also needs a short written note of every cleaning <!-- plain-language: everyday decision -->decision and its size in rows and pounds.

**[Definition of Done](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-definition-of-done)**: Tick these yourself before claiming any [badge](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-badge):

- [ ] Every ✏️ prediction cell in [First look](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-first-look) filled in *before* its code cell was run
- [ ] Both ledgers loaded and standardised to one schema (the same columns in both: `date`, `store`, `product`, `customer`, and `amount`)
- [ ] Every non-sale row and every duplicate dealt with, and *counted*, not dropped silently
- [ ] Combined monthly revenue reported for all six months, formatted in pounds
- [ ] The [Red Flags](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-red-flags) bug found, quantified in pounds, explained in business terms, and fixed
- [ ] Every [Debrief](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-debrief) prompt answered
- [ ] SpecStory transcript confirmed saved in `.specstory/history/`, and a dated entry in your `QuestLog.md` with the badge you are claiming

**Badge criteria this week:**

- 🥉 **[Bronze](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bronze)**: Both ledgers loaded, standardised, and stacked into one table. Monthly revenue table delivered and checked against one month you verified by hand
- 🥈 **[Silver](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-silver)**: Bronze, *plus* the two cleaning strategies judged with evidence of how many rows and how many pounds each <!-- plain-language: everyday decision -->decision moves. You defend your choice in writing
- 🥇 **[Gold](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-gold)**: Silver, *plus* an extension of your own design: Reconcile against a control total, or build a reusable data-quality report function

*Stage timings in minutes: Briefing 10, First look 15, Build 40, Red Flags 15, Debrief 10. They are guidance, not limits.*

### The data

Run the cell below first. It creates the files Finance sends you: Hartley's ledger, Voss's ledger, and the merged company's loyalty contact list (used in Red Flags). It uses the module seed (`5813`), so everyone sees the same data, mess included.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
# Hartley & Voss: Two sales ledgers that disagree, plus a loyalty contact file.
import os

import numpy as np
import pandas as pd

DATA_DIR = "assets/data/week04"
os.makedirs(DATA_DIR, exist_ok=True)
rng = np.random.default_rng(5813)  # module seed: Do not change

first_names = ["Amara", "Ben", "Chloe", "Dev", "Elena", "Farid", "Grace", "Hugo",
               "Isla", "Jack", "Kavya", "Liam", "Mei", "Noah", "Olive", "Priya",
               "Rosa", "Sam", "Tara", "Umar"]
last_names = ["Adeyemi", "Brooks", "Chen", "Dawson", "Evans", "Fletcher", "Gupta",
              "Hughes", "Iqbal", "Jones", "Khan", "Lawson", "Murphy", "Novak",
              "O'Brien", "Patel", "Quinn", "Reid", "Singh", "Turner"]
customers = sorted({f"{f} {l}" for f, l in
                    zip(rng.choice(first_names, 60), rng.choice(last_names, 60))})[:40]

hartley_products = {"Table Lamp": 34.00, "Cushion": 12.50, "Throw": 24.00,
                    "Candle Set": 9.75, "Mirror": 45.00, "Vase": 16.50,
                    "Photo Frame": 8.25, "Storage Basket": 14.00}
voss_products = {"Stock Pot": 42.00, "Chef's Knife": 55.00, "Mixing Bowl": 11.50,
                 "Cast Iron Pan": 65.00, "Measuring Jug": 6.75, "Serving Board": 18.00,
                 "Whisk": 7.50, "Teapot": 22.50}

all_days = pd.date_range("2026-01-01", "2026-06-30", freq="D")

# --- Hartley & Co: Title Case columns, DD/MM/YYYY dates, a TOTAL footer row,
# --- and 18 Amount values that went missing in the export (Qty/Unit Price intact).
n_h = 430
h_dates = rng.choice(all_days, n_h)
h_items = rng.choice(list(hartley_products), n_h)
h_qty = rng.integers(1, 6, n_h)
h_price = np.array([hartley_products[p] for p in h_items])
hartley = pd.DataFrame({
    "Date": pd.Series(h_dates).dt.strftime("%d/%m/%Y"),
    "Store": rng.choice(["Egham", "Staines", "Windsor"], n_h),
    "Product": h_items,
    "Customer": rng.choice(customers, n_h),
    "Qty": h_qty,
    "Unit Price": h_price,
    "Amount": np.round(h_qty * h_price, 2),
}).sort_values("Date", key=lambda s: pd.to_datetime(s, format="%d/%m/%Y"),
               ignore_index=True)
hartley_control_total = hartley["Amount"].sum()          # the TRUE Hartley total
missing_idx = rng.choice(hartley.index, 18, replace=False)
hartley.loc[missing_idx, "Amount"] = np.nan              # export mangled these
footer = pd.DataFrame([{"Date": "TOTAL", "Store": "", "Product": "", "Customer": "",
                        "Qty": np.nan, "Unit Price": np.nan,
                        "Amount": round(hartley_control_total, 2)}])
pd.concat([hartley, footer], ignore_index=True).to_csv(
    f"{DATA_DIR}/hartley_sales.csv", index=False)

# --- Voss Retail: snake_case columns, ISO dates, no amount column,
# --- and 14 rows accidentally exported twice.
n_v = 380
v_dates = rng.choice(all_days, n_v)
v_items = rng.choice(list(voss_products), n_v)
voss = pd.DataFrame({
    "txn_date": pd.Series(v_dates).dt.strftime("%Y-%m-%d"),
    "branch": rng.choice(["Kingston", "Richmond", "Guildford"], n_v),
    "item": v_items,
    "client_name": rng.choice(customers, n_v),
    "units": rng.integers(1, 7, n_v),
    "price_gbp": [voss_products[p] for p in v_items],
})
dup_rows = voss.sample(14, random_state=5813)
voss = (pd.concat([voss, dup_rows])
        .sample(frac=1, random_state=5813)     # shuffle the duplicates in
        .sort_values("txn_date", ignore_index=True))
voss.to_csv(f"{DATA_DIR}/voss_ledger.csv", index=False)

# --- Loyalty contacts: One row per member, except two names that belong
# --- to TWO different people each (same name, different member id and tier).
loyalty = pd.DataFrame({
    "member_id": [f"M{1000 + i}" for i in range(len(customers))],
    "customer_name": customers,
    "loyalty_tier": rng.choice(["Bronze", "Silver", "Gold"], len(customers)),
})
namesakes = pd.DataFrame({
    "member_id": ["M9001", "M9002"],
    "customer_name": [customers[7], customers[21]],   # second person, same name
    "loyalty_tier": ["Gold", "Gold"],
})
loyalty = pd.concat([loyalty, namesakes], ignore_index=True)
loyalty.to_csv(f"{DATA_DIR}/loyalty_contacts.csv", index=False)

for f in ["hartley_sales.csv", "voss_ledger.csv", "loyalty_contacts.csv"]:
    print(f"Created {DATA_DIR}/{f}")


## Stage 2: First look

*(about 15 minutes)*

Your Associate "had a first look overnight" and left the cells below. **[PREDICT-RUN](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-predict-run)**: Write one line in each ✏️ cell *before* you run the code below it. Then run it and note every surprise.

> ✏️ **Your prediction:** *(one line: Same shapes? Same column names? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 1: Load both ledgers, compare their shapes and schemas
import pandas as pd

h = pd.read_csv("assets/data/week04/hartley_sales.csv")
v = pd.read_csv("assets/data/week04/voss_ledger.csv")

print(f"Hartley: {h.shape[0]} rows x {h.shape[1]} cols: {list(h.columns)}")
print(f"Voss:    {v.shape[0]} rows x {v.shape[1]} cols: {list(v.columns)}")

> ✏️ **Your prediction:** *(one line: What will the LAST two rows of Hartley's ledger look like? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 2: Top and tail of the Hartley ledger
print(h.head(2))
print("...")
print(h.tail(2))

> ✏️ **Your prediction:** *(one line: Which `dtype` will each column have? Anything numeric that should not be, or the other way round? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 3: dtypes and duplicates. Trust nothing yet
print("Hartley dtypes:")
print(h.dtypes)
print("\nVoss dtypes:")
print(v.dtypes)
print(f"\nExact duplicate rows. Hartley: {h.duplicated().sum()}, Voss: {v.duplicated().sum()}")
print(f"Missing Amount values in Hartley: {h['Amount'].isna().sum()}")

> ✏️ **Your prediction:** *(one line: The Associate parses Hartley's dates with no format given, and this cell fails. What will the error message point to, and which row or date causes it? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 4: The Associate tries to parse Hartley's dates. (This cell is
# EXPECTED to fail. Read the error from the bottom up.)
pd.to_datetime(h["Date"])

**First look findings.** These problems are all real, and very common (remember: You'll spend a lot more time *cleaning* data than *analysing* data):

1. **The schemas disagree.** Hartley has `Date`, `Store`, `Product`, `Customer`, `Qty`, `Unit Price`, and `Amount`. Voss has `txn_date`, `branch`, `item`, `client_name`, `units`, and `price_gbp`. Voss has no amount column, so compute it as `units × price_gbp`
2. **Hartley's last row is not a sale.** It is a `TOTAL` footer. If you leave it in, adding up the amounts counts the whole ledger twice
3. **Hartley has 18 missing Amounts**, and **Voss has 14 exact duplicate rows**. In the Hartley rows with no Amount, `Qty` and `Unit Price` survived. Hartley's `Qty` reads as `float64` (decimals), not `int64` (whole numbers): The footer's empty cell forced the whole column to decimals
4. **Dates fail to parse.** Hartley writes `13/01/2026` (DD/MM/YYYY). pandas guessed month-first from the first row, then met `13` where it expected a month. Remove the footer, then state the format: [`pd.to_datetime(..., format="%d/%m/%Y")`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html). Voss writes its dates year first (`2026-01-13`), so they parse cleanly

Carry these findings into Build.

## Stage 3: Build

*(about 40 minutes)*

The glossary defines each tier: [Bronze](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bronze), [Silver](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-silver), and [Gold](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-gold). Ask your Associate to show you each step it took. <!-- values-guard: allow: a quest stage estimate, not the lecture length -->

### 🥉 Bronze

**Deliverable:** One table of combined monthly revenue, January to June 2026, in pounds. Build it from both ledgers, standardised into a single `DataFrame`.

This is a **[GENERATE-JUDGE](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-generate-judge)** job. Work through the steps below. One prompt per step works well. Tell your Associate what First look found.

1. **Standardise Hartley:** Remove the `TOTAL` footer row, parse `Date` with `format="%d/%m/%Y"`, and rename the columns to a shared schema: `date, store, product, customer, amount`. For the missing Amounts, either drop those rows or fill them from `Qty × Unit Price`, and note which you did
2. **Standardise Voss:** Drop the 14 exact duplicates, parse `txn_date`, compute `amount = units × price_gbp`, and rename to the same schema
3. **Combine:** Add a `company` column to each (`"Hartley"` or `"Voss"`), then stack them with [`pd.concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html). Count rows before and after
4. **Report:** Group by month and sum: `combined.groupby(combined["date"].dt.to_period("M"))["amount"].sum()`

**Check before you hand it over:**

- Six months, January to June, no gaps, and nothing from 2025 or July
- Hand-check one month: Filter both cleaned ledgers to that month, sum each, and add them. Does it equal your table's figure?
- Row arithmetic: Take Hartley's rows plus Voss's rows, then subtract the footer row, the duplicates, and any missing-Amount rows you dropped. Does that equal your combined row count? If not, where did the difference go?

In [ ]:
# 🥉 Bronze: Build it here with your Associate (add cells as needed).
# TODO 1: Standardise Hartley -> hartley_clean (drop footer, parse dates,
#         rename to: date, store, product, customer, amount)
# TODO 2: Standardise Voss -> voss_clean (drop duplicates, parse dates,
#         amount = units * price_gbp, same column names)
# TODO 3: combined = pd.concat([...], ignore_index=True)  (count rows!)
# TODO 4: Monthly revenue table, in pounds

pass

Put the hand-checked month and the row arithmetic in the cell below.

In [ ]:
# 🥉 Bronze: Verification (hand-check one month + row arithmetic).
# TODO: Prove one month's figure two ways, and reconcile your row counts.

pass

### 🥈 Silver

You asked your Associate what to do about Hartley's 18 missing Amounts, and it proposed **strategies that contradict each other**:

> **Strategy A: Drop suspect rows.** *"Missing revenue is untrustworthy revenue. Drop the 18 rows. Clean beats complete: You cannot defend numbers you invented."*
>
> **Strategy B: Repair and flag.** *"The export mangled `Amount`, but `Qty` and `Unit Price` survived: `Qty × Unit Price` *is* the amount, recomputed from source. Repair it, keep a `repaired` flag column, and disclose the repair. Dropping real sales understates revenue."*

Both run. Both produce a monthly table. The two totals differ by real money.

**Your Silver task:** Judge with evidence.

1. Run both strategy cells below
2. **Compute the stakes:** How many rows, and how many pounds, separate the two? The evidence cell below sets out the steps. Give a number, not an adjective
3. **Test your own choice:** Ask your Associate for the strongest case *against* the strategy you prefer. Then check the repair itself: On the rows where `Amount` survived, does `Qty × Unit Price` reproduce it?
4. **Choose and defend:** Write your choice and your reasoning in the markdown cell below. Use three or four sentences, and put the pounds in them

In [ ]:
# 🥈 Strategy A: Drop suspect rows (the Associate's first proposal). Runs as-is.
h_a = h[h["Date"] != "TOTAL"].copy()
h_a["date"] = pd.to_datetime(h_a["Date"], format="%d/%m/%Y")
h_a = h_a.dropna(subset=["Amount"])          # <- the decision: 18 rows gone
h_a = h_a.rename(columns={"Amount": "amount"})[["date", "amount"]]

v_a = v.drop_duplicates().copy()
v_a["date"] = pd.to_datetime(v_a["txn_date"], format="%Y-%m-%d")
v_a["amount"] = v_a["units"] * v_a["price_gbp"]
v_a = v_a[["date", "amount"]]

combined_a = pd.concat([h_a, v_a], ignore_index=True)
monthly_a = combined_a.groupby(combined_a["date"].dt.to_period("M"))["amount"].sum()
print("Strategy A, monthly revenue (suspect rows dropped):")
print(monthly_a.round(2))
print(f"H1 total: £{combined_a['amount'].sum():,.2f}  ({len(combined_a)} rows)")

Strategy B repairs the missing amounts from the surviving columns and flags every repaired row. It also runs as-is.

In [ ]:
# 🥈 Strategy B: Repair and flag (the Associate's second proposal). Runs as-is.
h_b = h[h["Date"] != "TOTAL"].copy()
h_b["date"] = pd.to_datetime(h_b["Date"], format="%d/%m/%Y")
h_b["repaired"] = h_b["Amount"].isna()       # <- flag before you fix
h_b["Amount"] = h_b["Amount"].fillna(h_b["Qty"] * h_b["Unit Price"])
h_b = h_b.rename(columns={"Amount": "amount"})[["date", "amount", "repaired"]]

v_b = v.drop_duplicates().copy()
v_b["date"] = pd.to_datetime(v_b["txn_date"], format="%Y-%m-%d")
v_b["amount"] = v_b["units"] * v_b["price_gbp"]
v_b["repaired"] = False
v_b = v_b[["date", "amount", "repaired"]]

combined_b = pd.concat([h_b, v_b], ignore_index=True)
monthly_b = combined_b.groupby(combined_b["date"].dt.to_period("M"))["amount"].sum()
print("Strategy B, monthly revenue (missing amounts repaired + flagged):")
print(monthly_b.round(2))
print(f"H1 total: £{combined_b['amount'].sum():,.2f}  ({len(combined_b)} rows, of which {combined_b['repaired'].sum()} repaired)")

The evidence cell below turns the disagreement into rows and pounds. Fill it in *before* you pick a side.

In [ ]:
# 🥈 The evidence: What is actually at stake between A and B?
# TODO: Rows at stake  (how many rows does A drop that B keeps?)
# TODO: Pounds at stake (difference between the two H1 totals, and which
#       months does it land in? monthly_b - monthly_a shows you)
# TODO: Trust check. On rows where Amount SURVIVED, does Qty * Unit Price
#       reproduce it? (If yes, the repair formula is trustworthy.)

pass

**✍️ My choice and defence:** *(A or B: Three or four sentences in your own words, with the rows and pounds in them. What would make you choose differently?)*

> 

### 🥇 Gold

The Partner reads over your shoulder: *"Good. Now make it stand up to the board's questions."* Design your own extension. Use one of these ideas, or bring your own:

- **Reconcile against a control total.** Hartley's footer row is *evidence*: It contains the grand total Hartley's own system computed before the export mangled anything. Extract it before you drop it. Which strategy, A or B, matches Hartley's own books to the penny? What does that tell the board about the other one?
- **A data-quality report function.** Write `quality_report(df)`. It returns or prints the row count, the duplicate count, missing values per column, and suspicious rows. Suspicious rows include dates that do not parse, and amounts that are negative or zero. Run it on both raw ledgers. Next month's export will be messy in the same ways

State your design in the opening comment of the cell below. Build it with your Associate. End with one sentence on what it proves.

In [ ]:
# 🥇 Gold. My extension: (state your design here in a comment)
# TODO: Build it with your Associate.

pass

## Stage 4: Red Flags

*(about 15 minutes)*

**[BUG-HUNT](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bug-hunt).** While you built the revenue table, the Associate drafted the loyalty analysis Finance asked for. It totals combined revenue by loyalty tier, using the merged company's contact list (`loyalty_contacts.csv`). It rebuilt the combined table correctly, then joined the loyalty file. The draft below **runs without an error** and prints a neat table.

Run it. Then compare its grand total with the H1 total (January to June) you trust from Build.

In [ ]:
# The Associate's draft: 'revenue by loyalty tier, ready for the board pack'.
# It runs clean. Is it right?
loyalty = pd.read_csv("assets/data/week04/loyalty_contacts.csv")

# Rebuild the combined table (footer out, dupes out, amounts repaired).
hh = h[h["Date"] != "TOTAL"].copy()
hh["date"] = pd.to_datetime(hh["Date"], format="%d/%m/%Y")
hh["amount"] = hh["Amount"].fillna(hh["Qty"] * hh["Unit Price"])
hh = hh.rename(columns={"Customer": "customer"})[["date", "customer", "amount"]]
vv = v.drop_duplicates().copy()
vv["date"] = pd.to_datetime(vv["txn_date"], format="%Y-%m-%d")
vv["amount"] = vv["units"] * vv["price_gbp"]
vv = vv.rename(columns={"client_name": "customer"})[["date", "customer", "amount"]]
combined_rf = pd.concat([hh, vv], ignore_index=True)

# Join loyalty tiers onto the sales, then total by tier.
with_tiers = combined_rf.merge(loyalty, left_on="customer",
                               right_on="customer_name", how="left")
tier_table = with_tiers.groupby("loyalty_tier")["amount"].sum().round(2)

print("Revenue by loyalty tier, H1 2026:")
print(tier_table)
print(f"\nGrand total: £{with_tiers['amount'].sum():,.2f}")

**Your audit:**

1. **Find it.** The grand total is *higher* than the H1 total you verified in Build, so the table invents revenue somewhere. Compare `len(combined_rf)` with `len(with_tiers)`. A left join adds loyalty columns to each sale, but it should not add rows. This one did. Find out why
   - *Stuck? Try these hints, one at a time:* How often does each customer **name** appear in the loyalty file? And why does the draft never touch the loyalty file's `member_id` column?
2. **Quantify and explain it in business terms** in the cell below. Phantom revenue is money the table shows but nobody spent. How many pounds of phantom revenue does the bug add? What does the error do to the <!-- plain-language: everyday decision -->decision? Do not stop at "the merge duplicated rows"
   - The board allocates marketing budget by tier from this table. Where several loyalty members share a name, every sale under that name is counted once for each of them
3. **Fix it by directing, not typing.** Tell your Associate what is wrong and why, and have *it* produce the corrected join. For example: *"This merge joins on customer name, but names are not unique. Some customer names belong to more than one loyalty member. Row counts jump from X to Y, and the total inflates by £Z. Fix the join so each sale matches at most one loyalty record, and show me how to prove the fix worked"*
   - *More than one fix is defensible,* because the sales ledger carries no member ID. Deduplicate name-tier pairs, match on a real key, or report the shared names as unmatched

In [ ]:
# 🚩 Quantify the damage: Make the phantom revenue a number.
# TODO: Rows before vs after the merge; pounds before vs after;
#       which customer names are duplicated in loyalty_contacts?

pass

**✍️ The bug, in business terms:** *(Two or three sentences: What is wrong, and how many pounds of phantom revenue? What would the board have done wrong because of it?)*

> 

Direct your Associate to produce the corrected join. Paste it into the cell below. The proof is one number: A grand total that matches the H1 figure you trust.

In [ ]:
# 🚩 Fixed analysis: Produced by directing your Associate, not by hand-editing.
# TODO: Paste the corrected join here, run it, and prove the grand total now
#       matches the H1 total you trust.

pass

## Stage 5: Debrief

*(about 10 minutes)*

**[TEACH-BACK](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-teach-back).** Answer each prompt in the cells below. Then copy your answers into your `QuestLog.md` as a dated entry with the badge you are claiming.

**1. Client explanation:** "Explain this week's key technique to the client in three sentences, no jargon." *(For Quest 04: What does it take to combine two companies' sales records, and why is it more than sticking the files together?)*

> 

**2. Associate audit:** "Where was your Associate wrong, misleading, or unhelpful this week? Paste the exchange." *(The Red Flags draft that joined on names counts, but an exchange from your own transcript is better. Quote its timestamp from `.specstory/history/`.)*

> 

**3. Quiz me:** "Ask your Associate to set you three questions on this week's material. Paste the quiz and answer it *without* AI help." *(Suggested prompt: "Set me three short questions (easy, medium, hard) on reading messy CSVs, `concat` vs `merge`, and why joining on names is dangerous. Don't show answers until I've tried.")*

> 

### 📼 Your record

From this week on, check your record after every session. Before you close VS Code, open `.specstory/history/`, and check that today's session file is there. Skim it once: Could a marker follow what you asked, what you rejected, and why? The [AI guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/ai-guide.md), section "6. Your record is your evidence", explains what <!-- plain-language: everyday abcd record -->the record is for.

Read the [Debrief](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/04/Debrief.ipynb) when it arrives with next week's materials. Its failure <!-- plain-language: everyday Gallery -->gallery includes a footer row that doubled a company's revenue.